# Scraping All Images from a Website with Gaffa

In this notebook, we'll use Gaffa to automatically discover, render, and download every image from a website.

**Use case:** You're building a computer vision model and need a dataset of space images. Instead of downloading them manually one by one, we'll automate the entire process using Gaffa — sitemap discovery, JavaScript rendering, and image downloading — all in a few lines of Python.

We'll scrape images from Gaffa's website.

## Step 1: Install Dependencies

In [1]:
!pip install requests

## Step 2: Set Up API Key and Headers

## *If you are using Google Colab: Run the cell below, then skip to Step 3.*

In [18]:
import os
import re
import requests
import json
from urllib.parse import urljoin
from google.colab import userdata

GAFFA_API_KEY = userdata.get("GAFFA_API_KEY")

if not GAFFA_API_KEY:
    raise RuntimeError("GAFFA_API_KEY is not set.")

HEADERS = {
    "x-api-key": GAFFA_API_KEY,
    "Content-Type": "application/json"
}

print("✅ API key loaded successfully.")

✅ API key loaded successfully.


## *If you are using an IDE and already have a `.env` file: Run the next two cells, then continue to Step 3.*

In [19]:
# !pip install python-dotenv

In [20]:
# import os
# import re
# import requests
# from urllib.parse import urljoin
# from dotenv import load_dotenv

# load_dotenv()

# GAFFA_API_KEY = os.getenv("GAFFA_API_KEY")

# if not GAFFA_API_KEY:
#     raise RuntimeError("GAFFA_API_KEY is not set.")

# HEADERS = {
#     "x-api-key": GAFFA_API_KEY,
#     "Content-Type": "application/json"
# }

# print("✅ API key loaded successfully.")

## Step 3: Discover All Pages Using Gaffa's Site Map Endpoint

Gaffa's `site/map` endpoint reads the site's sitemap and returns every available page URL automatically. This is the starting point — we don't need to manually find pages to scrape.

In [22]:
def get_sitemap_urls(site_url, max_cache_age=86400):
    payload = {
        "url": site_url,
        "max_cache_age": max_cache_age
    }

    print(f"Discovering pages on {site_url}...")
    response = requests.post(
        "https://api.gaffa.dev/v1/site/map",
        json=payload,
        headers=HEADERS
    )
    response.raise_for_status()

    urls = response.json()["data"]["links"]
    print(f"✅ Found {len(urls)} pages.")
    return urls

print("✅ get_sitemap_urls function defined.")

✅ get_sitemap_urls function defined.


## Step 4: Capture the Rendered DOM of Each Page

Many websites load images via JavaScript — a simple HTTP request won't see them. Gaffa spins up a real browser, executes the JavaScript, and captures the fully rendered HTML so we don't miss any images.

In [23]:
def get_dom(url):
    payload = {
        "url": url,
        "async": False,
        "settings": {
            "actions": [
                {
                    "type": "wait",
                    "selector": "img",
                    "timeout": 20000
                },
                {
                    "type": "capture_dom"
                }
            ],
            "time_limit": 40000
        }
    }

    print(f"Rendering page: {url}")
    response = requests.post(
        "https://api.gaffa.dev/v1/browser/requests",
        json=payload,
        headers=HEADERS
    )
    response.raise_for_status()

    dom_url = response.json()["data"]["actions"][1]["output"]
    dom_response = requests.get(dom_url)
    dom_response.raise_for_status()

    return dom_response.text

print("✅ get_dom function defined.")

✅ get_dom function defined.


## Step 5: Extract Image URLs from the DOM

Once we have the rendered HTML, we use a regex pattern to find all image `src` attributes — including lazy-loaded images that use `data-src`.

In [24]:
def extract_image_urls(dom_content, base_url):
    image_urls = []
    src_pattern = r'<img[^>]+(?:src|data-src)=["\']([^"\']+)["\']'
    matches = re.findall(src_pattern, dom_content)

    for src in matches:
        if not src.startswith(('http:', 'https:')):
            src = urljoin(base_url, src)
        image_urls.append(src)

    print(f"Found {len(image_urls)} images on this page.")
    return image_urls

print("✅ extract_image_urls function defined.")

✅ extract_image_urls function defined.


## Step 6: Download Images Using Gaffa

Instead of downloading images directly, we use Gaffa's `download_file` action. This routes the download through Gaffa's infrastructure with residential proxies and caching — so repeated runs won't hammer the target server and we won't get blocked.

In [25]:
def download_image(image_url, filename):
    payload = {
        "url": image_url,
        "async": False,
        "settings": {
            "actions": [
                {
                    "type": "download_file"
                }
            ]
        }
    }

    print(f"Downloading: {image_url}")
    response = requests.post(
        "https://api.gaffa.dev/v1/browser/requests",
        json=payload,
        headers=HEADERS
    )
    response.raise_for_status()

    download_url = response.json()["data"]["actions"][0]["output"]
    download_ext = os.path.splitext(download_url)[1] or ".jpg"

    img_response = requests.get(download_url)
    img_response.raise_for_status()

    filepath = f"{filename}{download_ext}"
    with open(filepath, 'wb') as f:
        f.write(img_response.content)

    print(f"✅ Saved to {filepath}")

print("✅ download_image function defined.")

✅ download_image function defined.


## Step 7: Run the Full Pipeline

Now let's put it all together. We'll:
1. Discover pages on Gaffa's website
2. Render each page and extract image URLs
3. Download the first image from each page using Gaffa

We limit to 3 pages to keep the demo fast — remove the `[:3]` slice to scrape the entire site.

In [26]:
os.makedirs("gaffa_images", exist_ok=True)

site_url = "https://gaffa.dev"
sitemap_urls = get_sitemap_urls(site_url)[:3]  # Limit to 3 pages for the demo

print(f"\nProcessing {len(sitemap_urls)} pages...\n")

for i, url in enumerate(sitemap_urls, 1):
    print(f"\n--- Page {i}: {url} ---")
    try:
        dom_content = get_dom(url)
        image_urls = extract_image_urls(dom_content, url)

        if image_urls:
            download_image(image_urls[0], f"gaffa_images/image_{i}")
        else:
            print("No images found on this page.")
    except Exception as e:
        print(f"⚠️ Error processing {url}: {e}")

print("\n✅ Done! Check the gaffa_images folder for your downloaded images.")

Discovering pages on https://gaffa.dev...
✅ Found 59 pages.

Processing 3 pages...


--- Page 1: https://gaffa.dev/about ---
Rendering page: https://gaffa.dev/about
Found 6 images on this page.
Downloading: https://gaffa.dev/_next/static/media/logo-light.a2132c1c.svg
✅ Saved to gaffa_images/image_1.svg

--- Page 2: https://gaffa.dev/blog ---
Rendering page: https://gaffa.dev/blog
Found 10 images on this page.
Downloading: https://gaffa.dev/_next/static/media/logo-light.a2132c1c.svg
✅ Saved to gaffa_images/image_2.svg

--- Page 3: https://gaffa.dev/blog/convert-any-web-page-to-llm-ready-markdown-using-gaffa ---
Rendering page: https://gaffa.dev/blog/convert-any-web-page-to-llm-ready-markdown-using-gaffa
Found 6 images on this page.
Downloading: https://gaffa.dev/_next/static/media/logo-light.a2132c1c.svg
✅ Saved to gaffa_images/image_3.svg

✅ Done! Check the gaffa_images folder for your downloaded images.


## What We Just Built

In just a few lines of Python and Gaffa, we:

- **Discovered** every page on Gaffa's website automatically using the sitemap endpoint
- **Rendered** each page with a real browser so JavaScript-loaded images weren't missed
- **Extracted** all image URLs from the fully rendered HTML
- **Downloaded** images using Gaffa's infrastructure with residential proxies and caching

This same pipeline works on any website. Change the `site_url` to scrape images from any public site — product photos, stock images, news photos — and you have an instant image dataset.